# CONVOLUTIONAL 1D VAE TRAINING

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
import h5py

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

In [ ]:
BACKGROUND_FILE = "../data/datasets/convolutional/background_convolutional_dataset.h5"

with h5py.File(BACKGROUND_FILE, "r") as f:
    X_train = f["X_train"][:].astype(np.float32)
    X_val = f["X_val"][:].astype(np.float32)
    X_test = f["X_test"][:].astype(np.float32)

print(f'{"X_train":<10}: {X_train.shape}')
print(f'{"X_val":<10}: {X_val.shape}')
print(f'{"X_test":<10}: {X_test.shape}')

In [ ]:
CONV1D_VAE_MODELS_DIR = "../models/convolutional1d_VAE"
os.makedirs(CONV1D_VAE_MODELS_DIR, exist_ok=True)

MODEL_NAME = "conv1d_vae_0"

SAVE_WEIGHTS_PATH = os.path.join(CONV1D_VAE_MODELS_DIR, f"{MODEL_NAME}.weights.h5")
SAVE_ENCODER_PATH = os.path.join(CONV1D_VAE_MODELS_DIR, f"{MODEL_NAME}_encoder.keras")
SAVE_DECODER_PATH = os.path.join(CONV1D_VAE_MODELS_DIR, f"{MODEL_NAME}_decoder.keras")

In [ ]:
N_OBJ = 19
N_FEAT = 3
LATENT_DIM = 12
BETA = 1e-3

W_CURR = {"pt": 1.0, "eta": 1.0, "phi": 0.5}
PT_STD = X_train[:, :, 0].std() + 1e-8

def reconstruction_event_loss(y_true, y_pred, pt_std=PT_STD, weights=W_CURR):
    y_true = tf.reshape(y_true, (-1, N_OBJ, N_FEAT))
    y_pred = tf.reshape(y_pred, (-1, N_OBJ, N_FEAT))

    mask = tf.cast(tf.not_equal(y_true[:, :, 0:1], 0.0), tf.float32)
    n_present = tf.maximum(tf.reduce_sum(mask, axis=[1, 2]), 1.0)

    pt_true = y_true[:, :, 0:1]
    pt_pred = y_pred[:, :, 0:1]
    pt_loss = tf.square(pt_true - pt_pred) / (pt_std ** 2)

    eta_true = y_true[:, :, 1:2]
    eta_pred = y_pred[:, :, 1:2]
    eta_loss = tf.square((eta_true - eta_pred) / 3.0)

    phi_true = y_true[:, :, 2:3]
    phi_pred = y_pred[:, :, 2:3]
    dphi = tf.atan2(tf.sin(phi_true - phi_pred), tf.cos(phi_true - phi_pred))
    phi_loss = tf.square(dphi)

    total = (
        weights["pt"] * pt_loss
        + weights["eta"] * eta_loss
        + weights["phi"] * phi_loss
    )
    total *= mask
    return tf.reduce_sum(total, axis=[1, 2]) / n_present

In [ ]:
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        eps = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * eps


def build_encoder(input_shape=(19, 3), latent_dim=12):
    inp = keras.Input(shape=input_shape, name="encoder_input")
    x = layers.BatchNormalization()(inp)
    x = layers.Conv1D(32, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)
    x = layers.Conv1D(64, 3, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)
    x = layers.Conv1D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)
    x = layers.Flatten()(x)

    z_mean = layers.Dense(latent_dim, name="z_mean")(x)
    z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
    z = Sampling(name="z")([z_mean, z_log_var])

    return keras.Model(inp, [z_mean, z_log_var, z], name="encoder")


def build_decoder(output_shape=(19, 3), latent_dim=12):
    latent_inputs = keras.Input(shape=(latent_dim,), name="decoder_input")
    x = layers.Dense(10 * 64, use_bias=False)(latent_inputs)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)
    x = layers.Reshape((10, 64))(x)

    x = layers.UpSampling1D(2)(x)
    x = layers.Conv1D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)

    x = layers.Conv1D(32, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)

    x = layers.Cropping1D(cropping=(0, 1))(x)
    out = layers.Conv1D(output_shape[1], 1, padding="same", name="reconstruction")(x)

    return keras.Model(latent_inputs, out, name="decoder")

In [ ]:
class Conv1DVAE(keras.Model):
    def __init__(self, encoder, decoder, beta=1e-3, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.beta = beta

        self.total_loss_tracker = keras.metrics.Mean(name="loss")
        self.reco_loss_tracker = keras.metrics.Mean(name="reco_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.reco_loss_tracker, self.kl_loss_tracker]

    def call(self, inputs, training=False):
        z_mean, z_log_var, z = self.encoder(inputs, training=training)
        reconstruction = self.decoder(z, training=training)
        return reconstruction

    def compute_losses(self, x, reconstruction, z_mean, z_log_var):
        reco_evt = reconstruction_event_loss(x, reconstruction)
        reco_loss = tf.reduce_mean(reco_evt)

        kl_evt = -0.5 * tf.reduce_sum(
            1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
            axis=1,
        )
        kl_loss = tf.reduce_mean(kl_evt)

        total_loss = reco_loss + self.beta * kl_loss
        return total_loss, reco_loss, kl_loss

    def train_step(self, data):
        x = data[0] if isinstance(data, tuple) else data

        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(x, training=True)
            reconstruction = self.decoder(z, training=True)
            total_loss, reco_loss, kl_loss = self.compute_losses(x, reconstruction, z_mean, z_log_var)

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.total_loss_tracker.update_state(total_loss)
        self.reco_loss_tracker.update_state(reco_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            "loss": self.total_loss_tracker.result(),
            "reco_loss": self.reco_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

    def test_step(self, data):
        x = data[0] if isinstance(data, tuple) else data
        z_mean, z_log_var, z = self.encoder(x, training=False)
        reconstruction = self.decoder(z, training=False)
        total_loss, reco_loss, kl_loss = self.compute_losses(x, reconstruction, z_mean, z_log_var)

        self.total_loss_tracker.update_state(total_loss)
        self.reco_loss_tracker.update_state(reco_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            "loss": self.total_loss_tracker.result(),
            "reco_loss": self.reco_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

In [ ]:
encoder = build_encoder(input_shape=(N_OBJ, N_FEAT), latent_dim=LATENT_DIM)
decoder = build_decoder(output_shape=(N_OBJ, N_FEAT), latent_dim=LATENT_DIM)
conv1d_vae = Conv1DVAE(encoder=encoder, decoder=decoder, beta=BETA, name="conv1d_vae")

conv1d_vae.compile(optimizer=keras.optimizers.Adam(1e-3))

_ = conv1d_vae(tf.zeros((1, N_OBJ, N_FEAT), dtype=tf.float32))

encoder.summary()
decoder.summary()

In [ ]:
EPOCHS = 100
BATCH_SIZE = 1024

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, verbose=1),
    keras.callbacks.ModelCheckpoint(
        SAVE_WEIGHTS_PATH,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
]

history_c1d_vae = conv1d_vae.fit(
    X_train,
    validation_data=(X_val,),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

conv1d_vae.load_weights(SAVE_WEIGHTS_PATH)
conv1d_vae.encoder.save(SAVE_ENCODER_PATH)
conv1d_vae.decoder.save(SAVE_DECODER_PATH)

print(f"Saved best weights: {SAVE_WEIGHTS_PATH}")
print(f"Saved encoder: {SAVE_ENCODER_PATH}")
print(f"Saved decoder: {SAVE_DECODER_PATH}")

In [ ]:
def plot_training_history(history, title="Conv1D VAE training"):
    fig, axs = plt.subplots(1, 3, figsize=(15, 4))

    axs[0].semilogy(history.history["loss"], label="Train")
    axs[0].semilogy(history.history["val_loss"], label="Validation")
    axs[0].set_title("Total loss")
    axs[0].set_xlabel("Epoch")
    axs[0].legend()

    axs[1].semilogy(history.history["reco_loss"], label="Train")
    axs[1].semilogy(history.history["val_reco_loss"], label="Validation")
    axs[1].set_title("Reconstruction")
    axs[1].set_xlabel("Epoch")
    axs[1].legend()

    axs[2].semilogy(history.history["kl_loss"], label="Train")
    axs[2].semilogy(history.history["val_kl_loss"], label="Validation")
    axs[2].set_title("KL divergence")
    axs[2].set_xlabel("Epoch")
    axs[2].legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_training_history(history_c1d_vae)